In [1]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.preprocess import preprocess
import json

In [2]:
raw_df = pd.read_csv("../data/raw.csv")

print("Кількість текстів:", len(raw_df))
raw_df.head()

Кількість текстів: 11161


,text_id,text
0,text_0,"Дорогою зупинялися в селах , старцювали . Усти..."
1,text_1,"То молитви продавав свої спудейські , то щось ..."
2,text_2,"Дуже часто дід досить точно вказував , куди са..."
3,text_3,"Часом Амфібрахій , керуючись настановами старо..."
4,text_4,Чи вірив у все це сам Амфібрахій ?


In [3]:
processed_rows = []

for _, row in raw_df.iterrows():
    result = preprocess(row["text"])

    processed_rows.append({
        "text_id": row["text_id"],
        "text_clean": result["clean"],
        "sentences": result["sentences"]
    })

processed_df = pd.DataFrame(processed_rows)

processed_df.head()

,text_id,text_clean,sentences
0,text_0,"Дорогою зупинялися в селах, старцювали. Устимк...","[Дорогою зупинялися в селах, старцювали., Усти..."
1,text_1,"То молитви продавав свої спудейські, то щось к...","[То молитви продавав свої спудейські, то щось ..."
2,text_2,"Дуже часто дід досить точно вказував, куди сам...","[Дуже часто дід досить точно вказував, куди са..."
3,text_3,"Часом Амфібрахій, керуючись настановами старог...","[Часом Амфібрахій, керуючись настановами старо..."
4,text_4,Чи вірив у все це сам Амфібрахій?,[Чи вірив у все це сам Амфібрахій?]


In [4]:
processed_df.to_csv("../data/processed_v2.csv", index=False)

In [5]:
raw_df["length_raw"] = raw_df["text"].str.len()
processed_df["length_processed"] = processed_df["text_clean"].str.len()

print("Середня довжина RAW:", raw_df["length_raw"].mean())
print("Середня довжина PROCESSED:", processed_df["length_processed"].mean())

same_ratio = (raw_df["text"] == processed_df["text_clean"]).mean()
print("Частка текстів без змін:", same_ratio)

Середня довжина RAW: 116.19120150524147
Середня довжина PROCESSED: 113.19917570110205
Частка текстів без змін: 0.04829316369500941


In [6]:
empty_raw = (raw_df["text"].str.strip() == "").sum()
empty_processed = (processed_df["text_clean"].str.strip() == "").sum()

print("Порожні RAW:", empty_raw)
print("Порожні PROCESSED:", empty_processed)

Порожні RAW: 0
Порожні PROCESSED: 0


In [7]:
short_raw = (raw_df["length_raw"] < 20).mean()
short_processed = (processed_df["length_processed"] < 20).mean()

print("Частка коротких RAW:", short_raw)
print("Частка коротких PROCESSED:", short_processed)

Частка коротких RAW: 0.05985126780754413
Частка коротких PROCESSED: 0.06379356688468775


In [8]:
raw_df["length_raw"].describe()

count    11161.000000
mean       116.191202
std         98.741364
min          1.000000
25%         50.000000
50%         90.000000
75%        151.000000
max       1310.000000
Name: length_raw, dtype: float64

In [9]:
processed_df["length_processed"].describe()

count    11161.000000
mean       113.199176
std         96.884632
min          1.000000
25%         49.000000
50%         88.000000
75%        147.000000
max       1263.000000
Name: length_processed, dtype: float64

In [10]:
url_count = processed_df["text_clean"].str.count("<URL>").sum()
email_count = processed_df["text_clean"].str.count("<EMAIL>").sum()
phone_count = processed_df["text_clean"].str.count("<PHONE>").sum()

print("URL замінено:", url_count)
print("Email замінено:", email_count)
print("Phone замінено:", phone_count)

URL замінено: 0
Email замінено: 3
Phone замінено: 12


In [11]:
edge_cases = []
with open("../tests/edge_cases.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        edge_cases.append(json.loads(line))

for case in edge_cases:
    result = preprocess(case["raw_text"])
    print(case["id"])
    print("RAW:", case["raw_text"])
    print("CLEAN:", result["clean"])
    print("-"*50)

text_5
RAW: Мабуть , ні .
CLEAN: Мабуть, ні.
--------------------------------------------------
text_39
RAW: Передасте ? »
CLEAN: Передасте?"
--------------------------------------------------
text_93
RAW: 3 .
CLEAN: 3.
--------------------------------------------------
text_54
RAW: Так запрацювала їхня « пошта » .
CLEAN: Так запрацювала їхня "пошта".
--------------------------------------------------
text_72
RAW: PR загалом покликаний забезпечувати ефективний діалог із суспільством , формуючи і підтримуючи позитивний образ , репутацію компанії , її послуг і ключових співробітників .
CLEAN: PR загалом покликаний забезпечувати ефективний діалог із суспільством, формуючи і підтримуючи позитивний образ, репутацію компанії, її послуг і ключових співробітників.
--------------------------------------------------
text_45
RAW: Що це — наївність чи безмежна довіра до тих , хто несе слово правди ? « А може , щось йому ще переказати , тому вдівцеві Паляниці ? » — « Перекажіть , що у нас усе добре

In [12]:
for case in edge_cases:
    clean1 = preprocess(case["raw_text"])["clean"]
    clean2 = preprocess(clean1)["clean"]
    assert clean1 == clean2, f"Idempotence failed: {case['id']}"

for case in edge_cases:
    clean = preprocess(case["raw_text"])["clean"]
    assert clean.strip() != "", f"Empty explosion: {case['id']}"